# Monthly Temperature Derivation

Generate a monthly table of average temperature per station from the daily data, for use as a new fact table in Power BI.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../Capstone/clean data/dim_temp_clean.csv")

C:\Users\biauser\AppData\Local\Temp\ipykernel_4680\3186851481.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../Capstone/clean data/dim_temp_clean.csv")


In [3]:
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 833189 entries, 0 to 833188
Data columns (total 7 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Climate ID          833189 non-null  object 
 1   Date/Time           833189 non-null  object 
 2   Max Temp (°C)       828481 non-null  float64
 3   Min Temp (°C)       831117 non-null  float64
 4   Mean Temp (°C)      833189 non-null  float64
 5   Heat Deg Days (°C)  826411 non-null  float64
 6   Cool Deg Days (°C)  826411 non-null  float64
dtypes: float64(5), object(2)
memory usage: 44.5+ MB


In [4]:

# Fix key data types early
df["Climate ID"] = df["Climate ID"].astype(str)

In [6]:
cols = [
    "Climate ID",
    "Date/Time",
    "Mean Temp (°C)"
]

df_temp = df[cols].copy()

In [7]:
df_temp["Date/Time"] = pd.to_datetime(df_temp["Date/Time"], errors="coerce")

df_temp["Year"] = df_temp["Date/Time"].dt.year
df_temp["Month"] = df_temp["Date/Time"].dt.month

# Forma recomendada para BI
df_temp["YearMonth"] = df_temp["Date/Time"].dt.to_period("M").astype(str)

In [8]:
df_temp = df_temp.dropna(
    subset=["Mean Temp (°C)", "Year", "Month"]
)

### Monthly aggregation (notebook core)
📊 Clear methodological definition:

Monthly temperature = average of daily mean temperatures

In [9]:
monthly_temp = (
    df_temp
    .groupby(
        ["Climate ID", "Year", "Month", "YearMonth"],
        as_index=False
    )
    ["Mean Temp (°C)"]
    .mean()
)

In [10]:
monthly_temp = monthly_temp.rename(
    columns={"Mean Temp (°C)": "MeanMonthlyTemp_C"}
)

In [11]:
monthly_temp.head(10)

,Climate ID,Year,Month,YearMonth,MeanMonthlyTemp_C
0,8100300,1995,1,1995-01,-9.061290
1,8100300,1995,2,1995-02,-13.600000
2,8100300,1995,3,1995-03,-3.287097
3,8100300,1995,4,1995-04,0.960000
4,8100300,1995,5,1995-05,10.483871
5,8100300,1995,6,1995-06,17.320000
6,8100300,1995,7,1995-07,20.477419
7,8100300,1995,8,1995-08,18.396774
8,8100300,1995,9,1995-09,12.820000
9,8100300,1995,10,1995-10,10.119355


In [12]:
monthly_temp.describe()
monthly_temp["YearMonth"].nunique()
monthly_temp["Climate ID"].nunique()

136

In [13]:
monthly_temp[
    monthly_temp["Climate ID"] == "8204020"
].sort_values("YearMonth")

,Climate ID,Year,Month,YearMonth,MeanMonthlyTemp_C
11285,8204020,2021,9,2021-09,15.330000
11286,8204020,2021,10,2021-10,10.487097
11287,8204020,2021,11,2021-11,3.696667
11288,8204020,2021,12,2021-12,-1.306452
11289,8204020,2022,1,2022-01,-6.740000
11290,8204020,2022,2,2022-02,-4.500000
11291,8204020,2022,3,2022-03,0.020000
11292,8204020,2022,4,2022-04,4.936667
11293,8204020,2022,5,2022-05,11.358065
11294,8204020,2022,6,2022-06,15.143333


In [14]:
output_path = "../Capstone/clean data/monthly_temperature_by_station.csv"

monthly_temp.to_csv(
    output_path,
    index=False
)

print(f"Monthly temperature file saved at: {output_path}")

Monthly temperature file saved at: ../Capstone/clean data/monthly_temperature_by_station.csv


In [15]:
monthly_temp.info()
monthly_temp.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28609 entries, 0 to 28608
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Climate ID         28609 non-null  object 
 1   Year               28609 non-null  int32  
 2   Month              28609 non-null  int32  
 3   YearMonth          28609 non-null  object 
 4   MeanMonthlyTemp_C  28609 non-null  float64
dtypes: float64(1), int32(2), object(2)
memory usage: 894.2+ KB


,Climate ID,Year,Month,YearMonth,MeanMonthlyTemp_C
0,8100300,1995,1,1995-01,-9.061290
1,8100300,1995,2,1995-02,-13.600000
2,8100300,1995,3,1995-03,-3.287097
3,8100300,1995,4,1995-04,0.960000
4,8100300,1995,5,1995-05,10.483871


In [16]:
monthly_temp[["Climate ID", "YearMonth"]].drop_duplicates().shape

(28609, 2)